In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

data = pd.read_csv("framingham.csv")
data

,male,age,education,currentSmoker,cigsPerDay,BPMeds,prevalentStroke,prevalentHyp,diabetes,totChol,sysBP,diaBP,BMI,heartRate,glucose,TenYearCHD
0,1,39,4.0,0,0.0,0.0,0,0,0,195.0,106.0,70.0,26.97,80.0,77.0,0
1,0,46,2.0,0,0.0,0.0,0,0,0,250.0,121.0,81.0,28.73,95.0,76.0,0
2,1,48,1.0,1,20.0,0.0,0,0,0,245.0,127.5,80.0,25.34,75.0,70.0,0
3,0,61,3.0,1,30.0,0.0,0,1,0,225.0,150.0,95.0,28.58,65.0,103.0,1
4,0,46,3.0,1,23.0,0.0,0,0,0,285.0,130.0,84.0,23.10,85.0,85.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4235,0,48,2.0,1,20.0,NaN,0,0,0,248.0,131.0,72.0,22.00,84.0,86.0,0
4236,0,44,1.0,1,15.0,0.0,0,0,0,210.0,126.5,87.0,19.16,86.0,NaN,0
4237,0,52,2.0,0,0.0,0.0,0,0,0,269.0,133.5,83.0,21.47,80.0,107.0,0
4238,1,40,3.0,0,0.0,0.0,0,1,0,185.0,141.0,98.0,25.60,67.0,72.0,0


In [2]:
data.fillna(data.median(), inplace=True)
x = data[['male', 'age', 'currentSmoker', 'cigsPerDay', 'BPMeds', 'prevalentStroke', 'prevalentHyp', 'diabetes', 'totChol', 'sysBP', 'diaBP', 'BMI', 'heartRate', 'glucose']]
y = data['TenYearCHD']
x_train, x_test, y_train, y_test = train_test_split(x, y, random_state = 0, test_size=0.3)

In [3]:
classification = RandomForestClassifier(
    n_estimators=300,
    min_samples_leaf=10,
    max_depth=None,
    random_state=0,
    class_weight='balanced')
classification.fit(x_train, y_train)

predictions = classification.predict(x_test)

y_probs = classification.predict_proba(x_test)[:, 1]
fpr, tpr, thresholds = roc_curve(y_test, y_probs)
J = tpr - fpr
best_idx = np.argmax(J)
best_threshold = thresholds[best_idx]
print("Оптимальный порог:", best_threshold)

new_predictions = (y_probs >= best_threshold).astype(int)

Оптимальный порог: 0.37599163240326705


In [4]:
print(classification_report(y_test, predictions, target_names=['false', 'true']))
print()
print(confusion_matrix(y_test, new_predictions), "Confusion matrix")
print()
print("ROC AUC:", roc_auc_score(y_test, classification.predict_proba(x_test)[:,1]))
print()
print("Recall:", tpr[best_idx])
print()
print("FPR:", fpr[best_idx])

              precision    recall  f1-score   support

       false       0.89      0.84      0.86      1076
        true       0.32      0.42      0.36       196

    accuracy                           0.77      1272
   macro avg       0.60      0.63      0.61      1272
weighted avg       0.80      0.77      0.78      1272


[[686 390]
 [ 61 135]] Confusion matrix

ROC AUC: 0.7057507017676959

Recall: 0.6887755102040817

FPR: 0.362453531598513


In [5]:
joblib.dump(classification, 'tree_model.pkl')
joblib.dump(x.columns.tolist(), 'feature_names.pkl')
medians_dict = x_train.median().to_dict()
joblib.dump(medians_dict, 'medians.pkl')
joblib.dump(best_threshold, 'best_threshold.pkl')

['best_threshold.pkl']